<a href="https://colab.research.google.com/github/jaya-v07/media-engagement-pred-ML-model/blob/main/src/prediction-ml-model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Data Acquisition

We source the raw dataset directly from Kaggle using the `kagglehub` library. This keeps the notebook self-contained and reproducible — anyone re-running this notebook will always pull the latest published version of the dataset without needing to manually download or upload a CSV file.

**Step 1.1 — Install the Kaggle client library**

`kagglehub` provides a lightweight interface to Kaggle's dataset hosting service, handling authentication, caching, and file extraction under the hood.

In [ ]:
pip install kagglehub

**Step 1.2 — Download the dataset**

We fetch the *Social Media Engagement Dataset* (`subashmaster0411/social-media-engagement-dataset`) using `kagglehub.dataset_download()`. This call automatically resolves the latest version, downloads the archive, extracts it into a local cache directory, and returns the path so it can be loaded into pandas in the next step.

In [ ]:
import kagglehub
import pandas as pd

# 2. Download the latest version of the dataset
path = kagglehub.dataset_download("subashmaster0411/social-media-engagement-dataset")

print("Path to dataset files:", path)

100%|██████████| 1.30M/1.30M [00:00<00:00, 46.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/subashmaster0411/social-media-engagement-dataset/versions/1


**Step 1.3 — Verify the extracted contents**

Kaggle archives can occasionally contain multiple files or nested folders, so before loading anything we explicitly list the directory contents to confirm the exact CSV filename. This small defensive step avoids hard-coding a filename that might change between dataset versions.

In [ ]:
import os

# List all files in the downloaded directory to find the exact CSV filename
files = os.listdir(path)
print("Files in directory:", files)

Files in directory: ['Social Media Engagement Dataset.csv']


## 2. Loading the Dataset

With the filename confirmed, we load the CSV into a pandas DataFrame and preview the first five rows. This gives us a first look at the schema — column names, data types at a glance, and the general shape of a single record — before moving on to a formal structural audit.

In [ ]:
csv_file_name = files[0]
full_csv_path = os.path.join(path, csv_file_name)
df = pd.read_csv(full_csv_path)
df.head()

,post_id,timestamp,day_of_week,platform,user_id,location,language,text_content,hashtags,mentions,...,comments_count,impressions,engagement_rate,brand_name,product_name,campaign_name,campaign_phase,user_past_sentiment_avg,user_engagement_growth,buzz_change_rate
0,kcqbs6hxybia,2024-12-09 11:26:15,Monday,Instagram,user_52nwb0a6,"Melbourne, Australia",pt,Just tried the Chromebook from Google. Best pu...,#Food,NaN,...,701,18991,0.19319,Google,Chromebook,BlackFriday,Launch,0.0953,-0.3672,19.1
1,vkmervg4ioos,2024-07-28 19:59:26,Sunday,Twitter,user_ucryct98,"Tokyo, Japan",ru,Just saw an ad for Microsoft Surface Laptop du...,"#MustHave, #Food","@CustomerService, @BrandCEO",...,359,52764,0.05086,Microsoft,Surface Laptop,PowerRelease,Post-Launch,0.1369,-0.4510,-42.6
2,memhx4o1x6yu,2024-11-23 14:00:12,Saturday,Reddit,user_7rrev126,"Beijing, China",ru,What's your opinion about Nike's Epic React? ...,"#Promo, #Food, #Trending",NaN,...,643,8887,0.45425,Nike,Epic React,BlackFriday,Post-Launch,0.2855,-0.4112,17.4
3,bhyo6piijqt9,2024-09-16 04:35:25,Monday,YouTube,user_4mxuq0ax,"Lagos, Nigeria",en,Bummed out with my new Diet Pepsi from Pepsi! ...,"#Reviews, #Sustainable","@StyleGuide, @BrandSupport",...,743,6696,0.42293,Pepsi,Diet Pepsi,LaunchWave,Launch,-0.2094,-0.0167,-5.5
4,c9dkiomowakt,2024-09-05 21:03:01,Thursday,Twitter,user_l1vpox2k,"Berlin, Germany",hi,Just tried the Corolla from Toyota. Absolutely...,"#Health, #Travel","@BrandSupport, @InfluencerName",...,703,47315,0.08773,Toyota,Corolla,LocalTouchpoints,Launch,0.6867,0.0807,38.8


## 3. Structural Audit

Before any cleaning or modeling, it's important to understand the dataset at a structural level: how many records exist, what data type each column holds, and whether any values are missing. `df.info()` gives us this overview in a single call.

In [ ]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   post_id                  12000 non-null  object 
 1   timestamp                12000 non-null  object 
 2   day_of_week              12000 non-null  object 
 3   platform                 12000 non-null  object 
 4   user_id                  12000 non-null  object 
 5   location                 12000 non-null  object 
 6   language                 12000 non-null  object 
 7   text_content             12000 non-null  object 
 8   hashtags                 12000 non-null  object 
 9   mentions                 8059 non-null   object 
 10  keywords                 12000 non-null  object 
 11  topic_category           12000 non-null  object 
 12  sentiment_score          12000 non-null  float64
 13  sentiment_label          12000 non-null  object 
 14  emotion_type          

**Observation:** the dataset contains 12,000 records across 28 columns, and only the `mentions` column has missing values (8,059 non-null out of 12,000). Every other field is fully populated. This is explored further below.

Next, we generate descriptive statistics for all numeric columns and cross-check the null count column-by-column to quantify exactly where and how much data is missing.

In [ ]:
print(df.describe())
print("---------------------------------------")
print("\nNULL VALUES SUM COLUMN WISE: \n\n",df.isnull().sum())

       sentiment_score  toxicity_score  likes_count  shares_count  \
count     12000.000000    12000.000000  12000.00000  12000.000000   
mean          0.000553        0.503868   2490.72025   1007.167167   
std           0.583563        0.288198   1441.53253    575.072282   
min          -0.999800        0.000000      0.00000      0.000000   
25%          -0.503200        0.251400   1236.00000    510.000000   
50%          -0.006200        0.505950   2496.00000   1018.000000   
75%           0.513525        0.756200   3723.25000   1501.000000   
max           0.999900        0.999900   5000.00000   2000.000000   

       comments_count   impressions  engagement_rate  user_past_sentiment_avg  \
count     12000.00000  12000.000000     12000.000000             12000.000000   
mean        504.34575  49811.338500         0.278137                 0.001472   
std         288.68416  28930.289451         1.149206                 0.576627   
min           0.00000    130.000000         0.001880  

## 4. Visual Statistical Summary

To make the distribution of each numeric feature easier to scan, we re-render the descriptive statistics table with a background gradient. Darker shading highlights larger magnitudes within a column, which makes outliers and scale differences (e.g. `impressions` vs `sentiment_score`) visually obvious without having to compare raw numbers.

In [ ]:
# Unload the interactive table just for this summary view to keep it clean
%unload_ext google.colab.data_table

# Format to 4 decimal places and add a soft blue heat map to highlight magnitude
summary_table = df.describe().style\
    .format("{:.4f}")\
    .background_gradient(cmap='Blues', axis=0)

summary_table

The google.colab.data_table extension is not loaded.


,sentiment_score,toxicity_score,likes_count,shares_count,comments_count,impressions,engagement_rate,user_past_sentiment_avg,user_engagement_growth,buzz_change_rate
count,12000.0000,12000.0000,12000.0000,12000.0000,12000.0000,12000.0000,12000.0000,12000.0000,12000.0000,12000.0000
mean,0.0006,0.5039,2490.7202,1007.1672,504.3458,49811.3385,0.2781,0.0015,0.0010,0.7297
std,0.5836,0.2882,1441.5325,575.0723,288.6842,28930.2895,1.1492,0.5766,0.2899,57.7872
min,-0.9998,0.0000,0.0000,0.0000,0.0000,130.0000,0.0019,-0.9996,-0.4999,-99.9000
25%,-0.5032,0.2514,1236.0000,510.0000,253.0000,24716.5000,0.0491,-0.4960,-0.2484,-48.7000
50%,-0.0062,0.5060,2496.0000,1018.0000,503.0000,49674.0000,0.0806,0.0019,0.0028,0.9000
75%,0.5135,0.7562,3723.2500,1501.0000,755.0000,74815.0000,0.1631,0.5017,0.2507,50.1000
max,0.9999,0.9999,5000.0000,2000.0000,1000.0000,99997.0000,32.2117,0.9994,0.4999,99.9000


## 5. Key Findings & Engineering Notes

The section below documents the anomalies and design characteristics uncovered during this audit, along with the reasoning behind them.

* Analysis 1 : I looked at the raw numbers and noticed that likes_count, shares_count, and comments_count look completely linear. In real life, social media doesn't work this way—some posts get 0 likes and a few get 10k+.

The Engineering Culprit: Because this dataset is synthetic, the generator used an artificial uniform distribution. This created a massive structural flaw where a post could randomly get 5,000 likes but only 130 impressions. When calculating `Engagement Rate = (Likes + Comments + Shares ) / Impressions` , dividing by such a tiny number of impressions creates an impossible outlier of 3,221% (max = 32.21).

* Analysis 2: The mentions column contains 3,941 null (NaN) records, while all other columns are entirely populated.

Engineering Insight: This missingness is informative rather than corrupted. A null value implies a real-world user action: the post simply did not mention/tag another user.


##IMPORTANT NOTE

Observation: The summary statistics capture explicit negative values for metrics like sentiment_score, user_past_sentiment_avg, user_engagement_growth and buzz_change_rate.

Engineering Insight: * The sentiment columns utilize a native Polarity Scale (bounded between [-1.0, 1.0]), where -1.0 signifies maximum toxicity/negativity, 0.0 represents neutral prose, and 1.0 tracks highly positive sentiment.
The buzz_change_rate operates as a velocity vector (bounded between [-100%, 100%]). A negative value indicates a decay in content momentum and topical conversation over the tracked window.

## 6. Feature Engineering

With the raw data audited, we now translate the findings from Section 5 into usable model inputs. Three targeted fixes/features are engineered here, each addressing a specific issue surfaced earlier:

- **`mentions` → `has_mention`**: rather than treating the 3,941 nulls in `mentions` as missing data   to be dropped or imputed with a guess, we encode the null itself as signal (per the Analysis 2   insight) — a binary flag for whether the post tagged another user.
- **`text_content` → `text_length`**: raw text can't be fed directly into a tree-based model, so we   derive a simple, high-signal numeric proxy — character count — which models can split on directly.
- **`engagement_rate` → `is_viral`**: since `engagement_rate` is corrupted by the divide-by-tiny-  impressions outlier problem identified in Analysis 1 (max = 3,221%), we sidestep the distortion   entirely by binarizing around the median. This reframes the problem as a balanced binary   classification task ("did this post outperform the median post?") instead of a noisy regression.




In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
# Fix the informative missingness in the mentions column
df['mentions'] = df['mentions'].fillna('none')

# Feature 1: Create a binary flag for whether a post tagged someone
df['has_mention'] = (df['mentions'] != 'none').astype(int)

# Feature 2: Extract text length from the content (models love numeric lengths)
df['text_length'] = df['text_content'].str.len()

# Feature 3: Target Binarization (Our fix for the 32.21 outlier rate)
# Using the median (0.0806) to split into perfectly balanced 1s and 0s
median_rate = df['engagement_rate'].median()
df['is_viral'] = (df['engagement_rate'] > median_rate).astype(int)

## 7. Modeling Environment Setup

To compare a tree-based baseline against a gradient-boosted model, we install `xgboost` alongside scikit-learn's built-in classifiers. This gives us three model architectures of increasing complexity to benchmark against one another in Section 8.

In [41]:
pip install xgboost


## 8. Model Evaluation Framework

Rather than manually training and scoring each model one at a time, we define a single reusable function, `evaluate_feature_set()`, that standardizes the entire evaluation workflow for any given list of feature columns:

1. **Split** — an 80/20 train-test split, stratified on `is_viral` so the class balance is preserved    in both sets.
2. **Lineup** — three classifiers of increasing complexity are trained side-by-side: a single    `DecisionTreeClassifier` (baseline/interpretable), a `RandomForestClassifier` (bagged ensemble),    and an `XGBClassifier` (boosted ensemble).
3. **Process** — each model's predictions are scored with `classification_report`, capturing    Precision, Recall, and F1-Score for both the "Low Engagement" and "Viral" classes, plus overall    accuracy.
4. **Format** — all results are collected into a single tidy, multi-indexed DataFrame (Model × Class),    making it trivial to compare architectures and feature sets side-by-side later on.

Wrapping this logic in a function means every subsequent feature-set experiment (Sections 9 and 10) is a single, consistent, one-line call — reducing the risk of copy-paste errors between runs.

In [49]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

def evaluate_feature_set(df, feature_list, target_col='is_viral', test_size=0.2):
    """
    Splits data, trains three models, and constructs a detailed, structured
    report containing Precision, Recall, and F1-Scores for both classes.
    """
    print(f"🔬 Detailed Audit for Features: {feature_list}\n" + "="*60)

    # 1. Split
    X_subset = df[feature_list].copy()
    y_subset = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset, test_size=test_size, random_state=42, stratify=y_subset
    )

    # 2. Lineup
    models = {
        "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
        "XGBoost": XGBClassifier(max_depth=5, random_state=42)
    }

    detailed_rows = []

    # 3. Process
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        # Output report as a dictionary instead of plain text
        report_dict = classification_report(y_test, preds, output_dict=True)

        # Extract metrics cleanly for both low-engagement (0) and high-engagement (1)
        for cl in ['0', '1']:
            detailed_rows.append({
                "Model Architecture": name,
                "Target Class": "Low Engagement (0)" if cl == '0' else "Viral (1)",
                "Precision": f"{report_dict[cl]['precision']:.4f}",
                "Recall": f"{report_dict[cl]['recall']:.4f}",
                "F1-Score": f"{report_dict[cl]['f1-score']:.4f}",
                "Overall Accuracy": f"{report_dict['accuracy']:.4f}"
            })

    # 4. Format into a beautiful overview table
    report_df = pd.DataFrame(detailed_rows)
    return report_df.set_index(["Model Architecture", "Target Class"])

In [50]:
metadata_cols = [
    'sentiment_score', 'toxicity_score', 'has_mention',
    'text_length', 'user_past_sentiment_avg',
    'user_engagement_growth', 'buzz_change_rate'
]

evaluate_feature_set(df, metadata_cols)

🔬 Detailed Audit for Features: ['sentiment_score', 'toxicity_score', 'has_mention', 'text_length', 'user_past_sentiment_avg', 'user_engagement_growth', 'buzz_change_rate']


Precision  Recall F1-Score  \
Model Architecture Target Class                                    
Decision Tree      Low Engagement (0)    0.5158  0.5308   0.5232   
                   Viral (1)             0.5167  0.5017   0.5091   
Random Forest      Low Engagement (0)    0.5087  0.5133   0.5110   
                   Viral (1)             0.5088  0.5042   0.5065   
XGBoost            Low Engagement (0)    0.5078  0.5133   0.5106   
                   Viral (1)             0.5080  0.5025   0.5052   

                                      Overall Accuracy  
Model Architecture Target Class                         
Decision Tree      Low Engagement (0)           0.5162  
                   Viral (1)                    0.5162  
Random Forest      Low Engagement (0)           0.5088  
                   Viral (1)                    0.5088  
XGBoost            Low Engagement (0)           0.5079  
                   Viral (1)                    0.5079

In [51]:
metadata_cols = [
    'impressions'
]

evaluate_feature_set(df, metadata_cols)

🔬 Detailed Audit for Features: ['impressions']


Precision  Recall F1-Score  \
Model Architecture Target Class                                    
Decision Tree      Low Engagement (0)    0.8264  0.8333   0.8299   
                   Viral (1)             0.8319  0.8250   0.8285   
Random Forest      Low Engagement (0)    0.8280  0.8467   0.8372   
                   Viral (1)             0.8431  0.8242   0.8335   
XGBoost            Low Engagement (0)    0.8278  0.8375   0.8326   
                   Viral (1)             0.8356  0.8258   0.8307   

                                      Overall Accuracy  
Model Architecture Target Class                         
Decision Tree      Low Engagement (0)           0.8292  
                   Viral (1)                    0.8292  
Random Forest      Low Engagement (0)           0.8354  
                   Viral (1)                    0.8354  
XGBoost            Low Engagement (0)           0.8317  
                   Viral (1)                    0.8317

In [53]:
from sklearn.preprocessing import LabelEncoder

# 1. Create a clean copy of the data slice
df_geo_social = df[['platform', 'language']].copy()

# 2. Encode both columns so the models can read them
le_platform = LabelEncoder()
le_language = LabelEncoder()

df_geo_social['platform'] = le_platform.fit_transform(df_geo_social['platform'])
df_geo_social['language'] = le_language.fit_transform(df_geo_social['language'])

# 3. Add the target back to this specific slice
df_geo_social['is_viral'] = df['is_viral']

# 4. Call your detailed evaluation function!
evaluate_feature_set(df_geo_social, ['platform', 'language'])

🔬 Detailed Audit for Features: ['platform', 'language']


Precision  Recall F1-Score  \
Model Architecture Target Class                                    
Decision Tree      Low Engagement (0)    0.4929  0.6058   0.5436   
                   Viral (1)             0.4886  0.3767   0.4254   
Random Forest      Low Engagement (0)    0.4907  0.5058   0.4982   
                   Viral (1)             0.4901  0.4750   0.4824   
XGBoost            Low Engagement (0)    0.4963  0.5517   0.5225   
                   Viral (1)             0.4953  0.4400   0.4660   

                                      Overall Accuracy  
Model Architecture Target Class                         
Decision Tree      Low Engagement (0)           0.4913  
                   Viral (1)                    0.4913  
Random Forest      Low Engagement (0)           0.4904  
                   Viral (1)                    0.4904  
XGBoost            Low Engagement (0)           0.4958  
                   Viral (1)                    0.4958